<a href="https://colab.research.google.com/github/Sharmadipti/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sharmadipti/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### My answer

**Task type: Ranking**

My lane is a ranking problem because the goal is to prioritize pages that are more likely to need attention. Instead of only deciding whether a page is good or bad, the model can assign a score to pages and help order them from highest priority to lowest priority.

This is useful because the team cannot review every page at the same time. A ranked list helps the team focus first on the pages where the model identifies the strongest signals of decline or improvement opportunity.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Load the starter dataset from the GitHub repository

import pandas as pd

url = "https://raw.githubusercontent.com/Sharmadipti/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

df.head()

Rows: 30,000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### My answer

The target I would predict is whether a content page is likely to experience a meaningful decline in search impressions.

For this assignment, the target can be defined as a proxy label: whether the page's impressions decline by more than 20% month-over-month.

This is a defined rule rather than a directly observed business outcome. It gives us a measurable target that can be used to investigate which pages may need attention.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create the proxy target:
# 1 = impressions declined by more than 20%
# 0 = otherwise

df["is_declining"] = (
    df["impressions_last_30d"] < 0.8 * df["impressions_prev_30d"]
).astype(int)

print("Target distribution:")
print(df["is_declining"].value_counts())

print("\nTarget rate:")
print(df["is_declining"].mean())

df[[
    "content_id",
    "impressions_last_30d",
    "impressions_prev_30d",
    "is_declining"
]].head(10)

Target distribution:
is_declining
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.5420666666666667


,content_id,impressions_last_30d,impressions_prev_30d,is_declining
0,content_304f48230142,578,987,1
1,content_a1fb4e703a9e,2501,5915,1
2,content_9aa793d4d895,2382,6089,1
3,content_331d6c4de07b,3626,4206,0
4,content_d99b7a2d90ca,4211,6452,1
5,content_d4084a4bc775,617,1009,1
6,content_9a34b442b552,1,13,1
7,content_a63219c6e95a,636,632,0
8,content_5e6c160719bc,5696,13828,1
9,content_c27558df2b0c,252,356,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*
Success metric: Precision@50

Precision@50 measures how many of the top 50 pages selected by the model are actually declining. This is useful because the team has limited time and needs to review the highest-priority pages first. A higher Precision@50 means the ranked list contains more genuinely declining pages, making the review queue more useful for decision-support.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: One row represents one content page.

Each content page has a content_id and contains its search and performance signals, such as impressions, clicks, sessions, CTR, position, and recent impression trends. The target is_declining indicates whether that page's impressions have declined based on the defined rule.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
unit_cols = [
    'content_id',
    'impressions_last_30d',
    'impressions_prev_30d',
    'clicks_last_30d',
    'ctr',
    'avg_position',
    'is_declining'
]

display(df[unit_cols].head(10))


,content_id,impressions_last_30d,impressions_prev_30d,clicks_last_30d,ctr,avg_position,is_declining
0,content_304f48230142,578,987,2,0.76,10.6,1
1,content_a1fb4e703a9e,2501,5915,2,0.05,20.3,1
2,content_9aa793d4d895,2382,6089,1,0.09,36.5,1
3,content_331d6c4de07b,3626,4206,22,0.49,6.2,0
4,content_d99b7a2d90ca,4211,6452,10,0.13,44.0,1
5,content_d4084a4bc775,617,1009,0,0.03,8.5,1
6,content_9a34b442b552,1,13,0,0.00,7.0,1
7,content_a63219c6e95a,636,632,1,0.06,21.2,0
8,content_5e6c160719bc,5696,13828,9,0.09,46.0,1
9,content_c27558df2b0c,252,356,0,0.16,4.9,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


A fixed rule can identify declining pages using one threshold, but it cannot easily combine many different signals such as previous impressions, visible queries, rare-share, anonymous-share, top-query share, and CTR. These signals can interact in different ways across content pages. ML can learn these patterns from the data and produce a more useful priority score for decision-support. This makes the approach more flexible than relying on a single if-statement.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.